In [1]:
# ---- CELDA 1: Evaluación de detección — cargar GT y tu salida ----
import json, csv
import numpy as np
from pathlib import Path
from collections import defaultdict

REPO_ROOT = Path("../..").resolve()
CLIP = "SNMOT-146"

# Tu salida del pipeline
JSON_IN = REPO_ROOT / "data" / "tracking" / f"integrado_soccernet_{CLIP}.json"
with open(JSON_IN, encoding="utf-8") as fh:
    tu = json.load(fh)
tus_frames = tu["frames"]

# Ground truth de SoccerNet (formato MOT)
GT_FILE = REPO_ROOT / "data" / "soccernet" / "tracking" / "test" / CLIP / "gt" / "gt.txt"
gt_por_frame = defaultdict(list)   # frame (1-indexed) -> lista de (track_id, x, y, w, h)
with open(GT_FILE) as fh:
    for fila in csv.reader(fh):
        fr = int(fila[0]); tid = int(fila[1])
        x, y, w, h = float(fila[2]), float(fila[3]), float(fila[4]), float(fila[5])
        gt_por_frame[fr].append((tid, x, y, w, h))

print(f"Tu JSON: {len(tus_frames)} frames")
print(f"GT: {len(gt_por_frame)} frames anotados")
print(f"GT detecciones totales: {sum(len(v) for v in gt_por_frame.values())}")
print(f"Tus detecciones totales: {sum(len(fr) for fr in tus_frames)}")

# IMPORTANTE: tu JSON es 0-indexed (frame 0..749), el GT es 1-indexed (1..750)
# Al comparar, tu frame i corresponde al GT frame i+1
print("\nNota: tu frame 0 = GT frame 1 (ajuste de índice al comparar)")

Tu JSON: 750 frames
GT: 750 frames anotados
GT detecciones totales: 7453
Tus detecciones totales: 5826

Nota: tu frame 0 = GT frame 1 (ajuste de índice al comparar)


In [2]:
# ---- CELDA 2: IoU + clase de cada track del GT (desde gameinfo.ini) ----
import configparser

def iou(boxA, boxB):
    # boxes en (x, y, w, h)
    ax1, ay1, ax2, ay2 = boxA[0], boxA[1], boxA[0]+boxA[2], boxA[1]+boxA[3]
    bx1, by1, bx2, by2 = boxB[0], boxB[1], boxB[0]+boxB[2], boxB[1]+boxB[3]
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    if ix2 < ix1 or iy2 < iy1: return 0.0
    inter = (ix2-ix1)*(iy2-iy1)
    union = boxA[2]*boxA[3] + boxB[2]*boxB[3] - inter
    return inter/union if union > 0 else 0.0

def tu_bbox_a_xywh(bbox):
    x1, y1, x2, y2 = bbox
    return (x1, y1, x2-x1, y2-y1)

# Clase de cada track_id del GT, agrupada en categorías comparables con tu sistema
INFO_FILE = REPO_ROOT / "data" / "soccernet" / "tracking" / "test" / CLIP / "gameinfo.ini"
cfg = configparser.ConfigParser()
cfg.read(INFO_FILE)

def categoria_gt(desc):
    d = desc.lower()
    if "ball" in d: return "balon"
    if "referee" in d: return "arbitro"
    if "player" in d or "goalkeeper" in d: return "persona"  # jugador o portero
    return "otro"

clase_gt = {}   # track_id GT -> categoría
for key in cfg["Sequence"]:
    if key.startswith("trackletid_"):
        tid = int(key.split("_")[1])
        clase_gt[tid] = categoria_gt(cfg["Sequence"][key])

from collections import Counter
print("Clases del GT por categoría:", dict(Counter(clase_gt.values())))

# Categoría de cada detección TUYA
def categoria_tuya(rol):
    if rol == "balon": return "balon"
    if rol == "arbitro": return "arbitro"
    if rol in ("jugador", "portero"): return "persona"
    return "otro"

print("Mapeo listo: GT y tu sistema agrupados en {persona, arbitro, balon}")

Clases del GT por categoría: {'persona': 19, 'arbitro': 2, 'balon': 1}
Mapeo listo: GT y tu sistema agrupados en {persona, arbitro, balon}


In [3]:
# ---- CELDA 3: Detección EN BLOQUE (personas: jugadores+porteros+árbitros) ----
# Compara todas tus cajas de persona contra todas las del GT, sin distinguir clase.
IOU_TH = 0.5

def evaluar_deteccion(categorias_incluidas):
    """Cuenta TP, FP, FN comparando tus cajas vs GT para las categorías dadas."""
    TP = FP = FN = 0
    for i, tus_dets in enumerate(tus_frames):
        gt_frame = i + 1   # tu frame 0 = GT frame 1
        # Cajas del GT de las categorías incluidas
        gt_boxes = [(x,y,w,h) for (tid,x,y,w,h) in gt_por_frame.get(gt_frame, [])
                    if clase_gt.get(tid) in categorias_incluidas]
        # Tus cajas de las categorías incluidas
        tus_boxes = [tu_bbox_a_xywh(d["bbox"]) for d in tus_dets
                     if categoria_tuya(d.get("rol")) in categorias_incluidas]
        # Emparejar por IoU (greedy)
        gt_usado = [False]*len(gt_boxes)
        for tb in tus_boxes:
            mejor_iou, mejor_j = 0, -1
            for j, gb in enumerate(gt_boxes):
                if gt_usado[j]: continue
                v = iou(tb, gb)
                if v > mejor_iou: mejor_iou, mejor_j = v, j
            if mejor_iou >= IOU_TH:
                TP += 1; gt_usado[mejor_j] = True
            else:
                FP += 1
        FN += gt_usado.count(False)
    prec = TP/(TP+FP) if TP+FP>0 else 0
    rec = TP/(TP+FN) if TP+FN>0 else 0
    f1 = 2*prec*rec/(prec+rec) if prec+rec>0 else 0
    return TP, FP, FN, prec, rec, f1

print("=== DETECCIÓN EN BLOQUE (todas las personas) ===")
TP,FP,FN,prec,rec,f1 = evaluar_deteccion({"persona","arbitro"})
print(f"TP={TP}, FP={FP}, FN={FN}")
print(f"Precision: {prec:.3f} | Recall: {rec:.3f} | F1: {f1:.3f}")

=== DETECCIÓN EN BLOQUE (todas las personas) ===
TP=5228, FP=237, FN=1475
Precision: 0.957 | Recall: 0.780 | F1: 0.859


In [4]:
# ---- CELDA 4: Detección POR CLASE (persona/árbitro por IoU, balón por distancia) ----

def centro_xywh(b):
    return (b[0]+b[2]/2, b[1]+b[3]/2)

def evaluar_balon(dist_max=15):
    """El balón se evalúa por distancia entre centros, no IoU (objeto pequeño)."""
    TP = FP = FN = 0
    for i, tus_dets in enumerate(tus_frames):
        gt_frame = i + 1
        gt_boxes = [(x,y,w,h) for (tid,x,y,w,h) in gt_por_frame.get(gt_frame, [])
                    if clase_gt.get(tid) == "balon"]
        tus_boxes = [tu_bbox_a_xywh(d["bbox"]) for d in tus_dets
                     if categoria_tuya(d.get("rol")) == "balon"]
        gt_usado = [False]*len(gt_boxes)
        for tb in tus_boxes:
            ct = centro_xywh(tb)
            mejor_d, mejor_j = 1e9, -1
            for j, gb in enumerate(gt_boxes):
                if gt_usado[j]: continue
                cg = centro_xywh(gb)
                d = np.hypot(ct[0]-cg[0], ct[1]-cg[1])
                if d < mejor_d: mejor_d, mejor_j = d, j
            if mejor_d <= dist_max:
                TP += 1; gt_usado[mejor_j] = True
            else:
                FP += 1
        FN += gt_usado.count(False)
    prec = TP/(TP+FP) if TP+FP>0 else 0
    rec = TP/(TP+FN) if TP+FN>0 else 0
    f1 = 2*prec*rec/(prec+rec) if prec+rec>0 else 0
    return TP, FP, FN, prec, rec, f1

print("=== DETECCIÓN POR CLASE ===\n")
# personas y árbitros por IoU (correcto para objetos grandes)
for cat in ["persona", "arbitro"]:
    TP,FP,FN,prec,rec,f1 = evaluar_deteccion({cat})
    print(f"{cat.upper():10s}: P={prec:.3f} R={rec:.3f} F1={f1:.3f}  (TP={TP} FP={FP} FN={FN})  [IoU≥0.5]")

# balón por distancia entre centros (correcto para objeto pequeño)
TP,FP,FN,prec,rec,f1 = evaluar_balon(dist_max=15)
print(f"{'BALON':10s}: P={prec:.3f} R={rec:.3f} F1={f1:.3f}  (TP={TP} FP={FP} FN={FN})  [centros≤15px]")

print("\nNota: el balón se evalúa por distancia entre centros (≤15px) porque con")
print("IoU≥0.5 un objeto de ~16px casi nunca empareja, aunque la detección sea correcta.")

=== DETECCIÓN POR CLASE ===

PERSONA   : P=0.923 R=0.814 F1=0.865  (TP=4998 FP=419 FN=1139)  [IoU≥0.5]
ARBITRO   : P=0.000 R=0.000 F1=0.000  (TP=0 FP=48 FN=566)  [IoU≥0.5]
BALON     : P=0.809 R=0.389 F1=0.526  (TP=292 FP=69 FN=458)  [centros≤15px]

Nota: el balón se evalúa por distancia entre centros (≤15px) porque con
IoU≥0.5 un objeto de ~16px casi nunca empareja, aunque la detección sea correcta.
